**Imports**

In [2]:
from datasets import DatasetDict,Dataset, load_dataset
from transformers import AutoTokenizer,AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
from transformers import DataCollatorWithPadding

**Load** **Data**

In [3]:
dataset= load_dataset("syedkhalid0/Sentiment-Analysis")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/4.55k [00:00<?, ?B/s]

train_data.csv:   0%|          | 0.00/6.94M [00:00<?, ?B/s]

val_data.csv:   0%|          | 0.00/870k [00:00<?, ?B/s]

test_data.csv:   0%|          | 0.00/870k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/83989 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10499 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10499 [00:00<?, ? examples/s]

**Load Pre-trained Model**

In [4]:
model_path="bert-base-uncased"

In [5]:
#load model tokenizer
tokenizer= AutoTokenizer.from_pretrained(model_path)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [6]:
#load model with binary classification head

model=AutoModelForSequenceClassification.from_pretrained(
    model_path,
    num_labels=3
    )

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


**Set Trainable Parameters**

In [7]:
#freeze all base model parameters
for name,param in model.base_model.named_parameters():
  param.requires_grad=False

#unfreeze base model pooling layers
for name,param in model.base_model.named_parameters():
  if "pooler" in name:
    param.requires_grad=True

**Data Pre-processing**

In [8]:
#define text preprocessing
def preprocess_function(example):
  return tokenizer(example["text"],truncation=True)

#preprocess all datasets
tokenized_data= dataset.map(preprocess_function,batched=True)

Map:   0%|          | 0/83989 [00:00<?, ? examples/s]

Map:   0%|          | 0/10499 [00:00<?, ? examples/s]

Map:   0%|          | 0/10499 [00:00<?, ? examples/s]

In [9]:

#create data collator
data_collator= DataCollatorWithPadding(tokenizer=tokenizer)

**Define Evaluation Metrix**

In [11]:
import numpy as np
from sklearn.metrics import roc_auc_score,accuracy_score # Import roc_auc_score directly from sklearn


import numpy as np
from sklearn.metrics import roc_auc_score, accuracy_score

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    probabilities = np.exp(predictions) / np.exp(predictions).sum(-1, keepdims=True)
    auc = np.round(roc_auc_score(labels, probabilities, multi_class="ovr", average="macro"),3)

    predicted_classes = np.argmax(predictions, axis=1)
    acc = np.round(accuracy_score(labels, predicted_classes), 3)
    return {"accuracy": acc, "auc": auc}

**Training Parameters**

In [12]:
lr=2e-5
batch_size=16
num_epochs=3

training_args=TrainingArguments(
    output_dir="results",
    learning_rate=lr,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=num_epochs

    #logging_strategy="epoch",
    #eval_strategy="epoch",
    #save_strategy="epoch",
    #load_best_model_at_end=True

)

**Fine-tune Model**

In [13]:
trainer= Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data["train"],
    eval_dataset=tokenized_data["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [14]:
trainer.train()

Step,Training Loss
500,0.878016
1000,0.752493
1500,0.682180
2000,0.648727
2500,0.630134
3000,0.615188
3500,0.600909
4000,0.588519
4500,0.589526
5000,0.588099


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=15750, training_loss=0.5983473791697669, metrics={'train_runtime': 1515.0171, 'train_samples_per_second': 166.313, 'train_steps_per_second': 10.396, 'total_flos': 1.053865593012015e+16, 'train_loss': 0.5983473791697669, 'epoch': 3.0})

In [15]:
 eval_results = trainer.evaluate()
print(eval_results)

Training Loss,Validation Loss,Step,Accuracy,Auc
0.567328,0.530207,15750,0.772000,0.910000


{'eval_loss': 0.5302066802978516, 'eval_accuracy': 0.772, 'eval_auc': 0.91}


Labels:

0: Negative

1: Neutral

2: Positive

In [16]:
from transformers import pipeline

model_id = "edaUsha/Fine_Tuning_Bert_For_Sentiment_Anaysis"

clf = pipeline(
    "text-classification",   # task
    model=model_id
)

print(clf("I really loved this movie!"))
print(clf("This was the worst experience ever."))

config.json:   0%|          | 0.00/969 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/322 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

[{'label': 'LABEL_2', 'score': 0.9575083255767822}]
[{'label': 'LABEL_0', 'score': 0.9032609462738037}]


In [17]:
clf("The crowd's support was overwhelming")

[{'label': 'LABEL_2', 'score': 0.9397210478782654}]

In [19]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_id = "edaUsha/Fine_Tuning_Bert_For_Sentiment_Anaysis"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSequenceClassification.from_pretrained(model_id)

inputs = tokenizer("I really like this product.", return_tensors="pt")
outputs = model(**inputs)

print(outputs.logits)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tensor([[-1.1790, -1.5241,  3.0625]], grad_fn=<AddmmBackward0>)
